# DQL: GROUP BY

`GROUP BY` puts rows into groups and calculates summaries for each group.

Grouping changes the level of detail in the result. Instead of returning one row per employee, a grouped query can return one row per department, one row per job, or one row for each department and job combination. This is how detail data becomes summary data.


## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## GROUP BY

Use `groupBy` to collect rows into groups, then calculate summaries for each group.

In plain English: `GROUP BY` makes buckets of rows. After rows are grouped, aggregate functions such as `count`, `avg`, and `sum` calculate one answer per bucket.

Common uses:

* Count employees by department.
* Calculate average salary by job title.
* Find total salary by department.
* Build summary tables from transaction or event data.

Key points:

* Grouping by `deptno` creates one result row per department number.
* Every selected column must either be part of the group or be calculated with an aggregate function.
* This is how we answer questions like "how many employees are in each department?"
* Grouping can trigger a shuffle because Spark may need to move rows with the same group key to the same executor.

Watch out: after `GROUP BY`, the original row-level columns are no longer available unless they are included in the group key or summarized with an aggregate function.


In [ ]:
emp.groupBy("deptno").agg(
    F.count("*").alias("employee_count"),
    F.round(F.avg("sal"), 2).alias("avg_salary"),
    F.sum("sal").alias("total_salary")
).orderBy("deptno").show()

spark.sql("""
SELECT deptno, COUNT(*) AS employee_count, ROUND(AVG(sal), 2) AS avg_salary, SUM(sal) AS total_salary
FROM emp
GROUP BY deptno
ORDER BY deptno
""").show()
